# Notebook 09 – Adaptive Repository

Build the **final adaptive repository** consumed by the website.

This notebook combines (no LLM, no additional inference):

- Default repositories (Notebook 05)
- Persona overrides (Notebook 06)
- Mood overrides (Notebook 07)
- Trait modifiers (Notebook 08)

into one organized `adaptive_repository/` folder with lookup indexes and validation.


## Inputs
- `data/outputs/global_defaults.json`
- `data/outputs/desktop_defaults.json`
- `data/outputs/mobile_defaults.json`
- `data/outputs/persona_overrides.json`
- `data/outputs/mood_overrides.json`
- `data/outputs/trait_modifiers.json`

## Outputs
- `data/outputs/adaptive_repository/` — all repository JSON files + lookup indexes
- `reports/AdaptiveRepository/repository_summary.xlsx`
- `data/outputs/adaptive_repository/repository_statistics.json`


In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.adaptive_repository.pipeline import run_adaptive_repository_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("AdaptiveRepository")
INPUT_DIR = PATHS.data_outputs
REPOSITORY_DIR = INPUT_DIR / "adaptive_repository"

print(f"Input directory: {INPUT_DIR}")
print(f"Repository output: {REPOSITORY_DIR}")
print(f"Reports: {REPORTS}")


Input directory: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs
Repository output: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository
Reports: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/AdaptiveRepository


## Check Inputs


In [2]:
INPUTS = {
    "Global defaults": INPUT_DIR / "global_defaults.json",
    "Desktop defaults": INPUT_DIR / "desktop_defaults.json",
    "Mobile defaults": INPUT_DIR / "mobile_defaults.json",
    "Persona overrides": INPUT_DIR / "persona_overrides.json",
    "Mood overrides": INPUT_DIR / "mood_overrides.json",
    "Trait modifiers": INPUT_DIR / "trait_modifiers.json",
}

for label, path in INPUTS.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'}")


Global defaults: OK
Desktop defaults: OK
Mobile defaults: OK
Persona overrides: OK
Mood overrides: OK
Trait modifiers: OK


## Build and Validate Repository


In [3]:
result = run_adaptive_repository_pipeline(INPUT_DIR, REPOSITORY_DIR, REPORTS)

validation = result.validation
statistics = result.statistics
lookup = result.lookup_indexes

print(f"Validation: {'PASS' if validation.is_valid else 'FAIL'}")
if validation.missing_values:
    print(f"Missing values: {len(validation.missing_values)}")
if validation.duplicate_keys:
    print(f"Duplicate keys: {validation.duplicate_keys}")
if validation.invalid_ui_names:
    print(f"Invalid UI names: {len(validation.invalid_ui_names)}")
if validation.invalid_categories:
    print(f"Invalid categories: {len(validation.invalid_categories)}")


INFO: Exported adaptive repository to /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository


Validation: PASS


## Repository Statistics


In [4]:
print(f"Defaults — global: {statistics['defaults']['global']}, "
      f"desktop: {statistics['defaults']['desktop']}, "
      f"mobile: {statistics['defaults']['mobile']}")
print(f"Persona overrides: {statistics['persona_overrides']['total_overrides']} "
      f"({statistics['persona_overrides']['personas']} personas)")
print(f"Mood overrides: {statistics['mood_overrides']['total_overrides']} "
      f"({statistics['mood_overrides']['moods']} moods)")
print(f"Trait modifiers: {statistics['trait_modifiers']['total_nudges']} nudges "
      f"({statistics['trait_modifiers']['entries']} entries)")


Defaults — global: 14, desktop: 13, mobile: 14
Persona overrides: 40 (6 personas)
Mood overrides: 32 (8 moods)
Trait modifiers: 14 nudges (9 entries)


## Lookup Indexes (sample)


In [5]:
persona_sample = dict(list(lookup["persona"].items())[:2])
mood_sample = dict(list(lookup["mood"].items())[:2])
trait_sample = dict(list(lookup["trait"].items())[:2])

print("Persona lookup sample:")
print(json.dumps(persona_sample, indent=2, ensure_ascii=False))
print("\nMood lookup sample:")
print(json.dumps(mood_sample, indent=2, ensure_ascii=False))
print("\nTrait lookup sample:")
print(json.dumps(trait_sample, indent=2, ensure_ascii=False))


Persona lookup sample:
{
  "Browser": {
    "n_overrides": 7,
    "ui_elements": [
      "color_theme_pref",
      "desktop_grid_pref",
      "desktop_navigation",
      "desktop_price_display",
      "desktop_review_display",
      "form_field_style",
      "urgency_pref"
    ],
    "overrides": {
      "color_theme_pref": {
        "value": "Vibrant Bold (Bright, energetic colors - exciting and dynamic)",
        "confidence": 0.277778,
        "support": 0.044643,
        "evidence": [
          "Statistics",
          "Random Forest",
          "SHAP"
        ]
      },
      "urgency_pref": {
        "value": "None (I find these annoying and manipulative)",
        "confidence": 0.277778,
        "support": 0.044643,
        "evidence": [
          "Statistics",
          "Random Forest",
          "SHAP"
        ]
      },
      "form_field_style": {
        "value": "Outlined (Border around the field)",
        "confidence": 0.277778,
        "support": 0.044643,
        "eviden

## Exports


In [6]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")


Export locations:
- global_defaults_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository/global_defaults.json
- desktop_defaults_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository/desktop_defaults.json
- mobile_defaults_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository/mobile_defaults.json
- persona_overrides_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository/persona_overrides.json
- mood_overrides_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository/mood_overrides.json
- trait_modifiers_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/adaptive_repository/trait_modifiers.json
- lookup_indexes_json: /Users/mariam/Downloads/New_Master copy/sma

## Final Output


In [7]:
if result.summary["validation_passed"]:
    print("Repository validated successfully.")
else:
    print("Repository validation failed — see issues above.")

print("\nRepository statistics:")
print(json.dumps(statistics, indent=2, ensure_ascii=False))


Repository validated successfully.

Repository statistics:
{
  "version": "1.0",
  "defaults": {
    "global": 14,
    "desktop": 13,
    "mobile": 14,
    "total": 41
  },
  "persona_overrides": {
    "personas": 6,
    "total_overrides": 40,
    "by_persona": {
      "Browser": 7,
      "Deal Hunter": 8,
      "Impulsive Buyer": 4,
      "Loyal Customer": 10,
      "Minimalist": 5,
      "Researcher": 6
    }
  },
  "mood_overrides": {
    "moods": 8,
    "total_overrides": 32,
    "by_mood": {
      "Bored": 5,
      "Excited": 4,
      "Frustrated": 0,
      "Happy": 6,
      "Neutral": 5,
      "Relaxed": 6,
      "Sad": 0,
      "Stressed": 6
    }
  },
  "trait_modifiers": {
    "entries": 9,
    "total_nudges": 14,
    "by_trait": {
      "Extraversion": 3,
      "Agreeableness": 1,
      "Conscientiousness": 3,
      "Neuroticism": 4,
      "Openness": 3
    }
  }
}
